# S4_J1_Why_Agent_Frameworks.ipynb

# Semaine 4 — Construire un Framework d'Agents

## Jour 1 — Pourquoi les frameworks existent-ils ?

**Objectifs**

À la fin de ce notebook, vous saurez :

- expliquer pourquoi les frameworks d'agents sont apparus ;
- identifier les problèmes qu'ils résolvent ;
- distinguer logique métier, orchestration et infrastructure ;
- comprendre pourquoi LangChain, LangGraph, OpenAI Agents SDK, CrewAI ou Google ADK proposent des abstractions différentes.

> Fil rouge : construire progressivement notre propre framework d'agents avant d'utiliser ceux du marché.


# Plan

1. Le problème
2. Les limites d'un agent codé "à la main"
3. Les responsabilités d'un framework
4. Décomposer un framework
5. Premier squelette de framework
6. Exercices
7. Notes d'architecte
8. Synthèse


# 1. Le problème

Lorsque l'on débute avec l'API OpenAI, un agent est relativement simple :

```text
Utilisateur
    │
    ▼
LLM
    │
Décision
    │
Tool
    │
Résultat
```

Mais dès que plusieurs outils, plusieurs agents ou de la mémoire sont ajoutés, le code devient difficile à maintenir.


# 2. Que contient réellement un agent ?

Un agent n'est pas seulement un appel à un LLM.

Il combine généralement :

- un modèle ;
- un prompt système ;
- une mémoire ;
- une boucle de décision ;
- des outils ;
- une gestion d'erreurs ;
- un état de conversation ;
- parfois un orchestrateur multi-agent.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class AgentConfig:
    name: str
    model: str
    system_prompt: str
    tools: list = field(default_factory=list)

cfg = AgentConfig(
    name="assistant",
    model="gpt-5",
    system_prompt="Tu es un assistant IA."
)

cfg


# 3. Où se trouve la complexité ?

La plupart du code d'un projet d'agents n'est **pas** liée au prompt.

Elle concerne :

- la gestion de l'état ;
- les appels d'outils ;
- les erreurs ;
- la mémoire ;
- l'observabilité ;
- les transitions entre agents.


In [ ]:
class FakeTool:
    def __init__(self, name):
        self.name = name

    def run(self, query):
        return f"[{self.name}] -> {query}"

weather = FakeTool("weather")
print(weather.run("Paris"))


# 4. Responsabilités d'un framework

Un framework prend en charge une partie de la "plomberie".

| Responsabilité | Exemple |
|---|---|
| Enregistrer des outils | Tool Registry |
| Gérer le contexte | Memory |
| Exécuter des étapes | Graph / Workflow |
| Tracer les appels | Observabilité |
| Gérer l'état | State |
| Relancer après erreur | Retry |


# 5. Première architecture

```text
            Agent
              │
      ┌───────┴────────┐
      │                │
   Memory         Tool Registry
      │                │
      └───────┬────────┘
              │
         LLM Adapter
```


In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools = {}

    def register(self, name, func):
        self.tools[name] = func

    def execute(self, name, *args, **kwargs):
        return self.tools[name](*args, **kwargs)


registry = ToolRegistry()

registry.register("echo", lambda text: text.upper())

registry.execute("echo", "bonjour")


In [ ]:
class Memory:
    def __init__(self):
        self.messages=[]

    def add(self, role, content):
        self.messages.append((role,content))

    def history(self):
        return self.messages

mem = Memory()
mem.add("user","Bonjour")
mem.add("assistant","Bonjour !")
mem.history()


In [ ]:
class MiniAgent:
    def __init__(self, registry, memory):
        self.registry=registry
        self.memory=memory

    def use_tool(self, tool, value):
        result=self.registry.execute(tool,value)
        self.memory.add("tool",result)
        return result

agent=MiniAgent(registry,mem)
agent.use_tool("echo","framework")


# 6. Réflexion

Notre mini framework contient déjà :

- Tool Registry
- Memory
- Agent

Les frameworks industriels ajoutent ensuite :

- Graphes d'exécution
- Multi-agent
- Handoffs
- Persistance
- Observabilité
- Guardrails


# 7. Comparaison conceptuelle

| Notre code | LangGraph | OpenAI Agents SDK |
|---|---|---|
| ToolRegistry | Tools | Tools |
| Memory | State | Session |
| MiniAgent | Node / Graph | Agent |
| use_tool() | Edge | Runner |


# 8. Exercice 1

Refactorisez `MiniAgent` pour :

- enregistrer plusieurs outils ;
- conserver un historique complet ;
- ajouter une méthode `chat()` (sans appel LLM pour l'instant).


In [ ]:
# Votre implémentation ici


# 9. Exercice 2

Listez les éléments qui relèvent :

- de la logique métier ;
- de l'orchestration ;
- de l'infrastructure.

Utilisez votre agent développé en semaine 2 comme référence.


In [ ]:
# Notes personnelles


# 🧠 Notes d'architecte

Pourquoi OpenAI a-t-il créé son Agent SDK alors que LangChain existait ?

Réponse courte :

- intégration native avec les API OpenAI ;
- moins d'abstractions ;
- meilleur contrôle de l'orchestration ;
- compatibilité directe avec Responses API, Tools et Handoffs.

À retenir : un framework est un compromis entre simplicité, flexibilité et dépendance à un écosystème.


# Synthèse

À retenir :

- Les frameworks ne créent pas de nouveaux concepts.
- Ils encapsulent des problèmes récurrents.
- Comprendre les concepts avant le framework permet de mieux choisir les bons outils.
- Dès demain, nous commencerons à construire un mini framework plus complet, qui servira de point de comparaison avec LangChain et LangGraph.
